In [ ]:
# Install packages
!pip install ortools         # Google OR-Tools for VRP optimization
!pip install requests        # For calling OpenRouteService API
!pip install folium          # For map visualization

In [ ]:
#Imports
import random  # To generate random student locations near the school
import requests  # To call the OpenRouteService API for driving times
import numpy as np  # To handle and process the duration matrix
import folium  # To create interactive maps showing school and student stops
from ortools.constraint_solver import  pywrapcp #Python wrapper that provides RoutingIndexManager and RoutingModel
from ortools.constraint_solver import routing_enums_pb2

In [ ]:
#Configuration
# School coordinates (Mansoura, Egypt)
school_lat = 31.0364
school_lon = 31.3807

# Number of students and buses
num_students = 10
num_buses = 2
vehicle_capacities = [5, 5]  # capacity for each bus

# Radius for generating student locations (~1 km)
max_offset_deg = 0.01

# OpenRouteService API key
from getpass import getpass
ORS_API_KEY = getpass("Enter your OpenRouteService API key: ")

In [ ]:
# Simulate Student Locations
# Set random seed for reproducibility
random.seed(42)
#seed is starting value for the random generator

# Generate random student GPS coordinates near the school
student_locations = []
for _ in range(num_students):
    lat_offset = random.uniform(-max_offset_deg, max_offset_deg) #Picks a random number between -0.01 and +0.01 degrees / north or south
    lon_offset = random.uniform(-max_offset_deg, max_offset_deg) #Picks a random number between -0.01 and +0.01 degrees / east or west
    student_lat = school_lat + lat_offset
    student_lon = school_lon + lon_offset
    student_locations.append((student_lat, student_lon))

# Combine school + students into locations list
locations = [(school_lat, school_lon)] + student_locations

# Print today's simulated pickup locations
print("Today's simulated pickup locations:")
for idx, (lat, lon) in enumerate(locations):
    label = "School" if idx == 0 else f"Student {idx}"
    print(f"{label}: ({lat:.6f}, {lon:.6f})")

Today's simulated pickup locations:
School: (31.036400, 31.380700)
Student 1: (31.039189, 31.371200)
Student 2: (31.031901, 31.375164)
Student 3: (31.041129, 31.384234)
Student 4: (31.044244, 31.372439)
Student 5: (31.034838, 31.371296)
Student 6: (31.030773, 31.380807)
Student 7: (31.026931, 31.374677)
Student 8: (31.039398, 31.381599)
Student 9: (31.030809, 31.382485)
Student 10: (31.042589, 31.370830)


In [ ]:
#duration matrix is a 2D table where each cell tells you the estimated travel time between two locations
#Build Duration Matrix with OpenRouteService
coordinates = [[lon, lat] for lat, lon in locations]

ors_url = "https://api.openrouteservice.org/v2/matrix/driving-car"
ors_headers = {
    "Authorization": ORS_API_KEY,
    "Content-Type": "application/json"
}
#This configures a POST request to the ORS Matrix API for car travel durations.
#A POST request is how your program sends data to a server and asks it to process or compute something
ors_body = {
    "locations": coordinates,
    "metrics": ["duration"],  # request driving time matrix
    "units": "m"
}

try:
    response = requests.post(ors_url, headers=ors_headers, json=ors_body, timeout=30) #send request
    if response.status_code == 200:
        result = response.json()
        durations = np.array(result["durations"])
        duration_matrix_minutes = (durations / 60).astype(int)
        print("Duration matrix (minutes):")
        print(duration_matrix_minutes)
    else:
        print("Error fetching duration matrix:", response.status_code)
        print(response.text)
        raise RuntimeError("Failed to retrieve duration matrix from OpenRouteService API.")
except requests.exceptions.RequestException as e:
    print("Exception during API request:", e)
    raise

Duration matrix (minutes):
[[0 1 1 2 2 2 2 2 0 2 2]
 [1 0 2 2 1 1 2 2 1 3 1]
 [1 1 0 2 2 1 1 1 1 1 2]
 [2 2 3 0 2 3 3 3 1 4 3]
 [2 1 3 3 0 2 4 3 1 4 1]
 [2 1 2 3 2 0 2 2 1 2 2]
 [1 2 2 2 4 3 0 1 1 0 3]
 [2 2 1 3 4 2 1 0 2 1 4]
 [2 2 3 1 2 3 3 3 0 3 3]
 [1 2 2 2 3 3 0 1 1 0 3]
 [2 1 3 3 1 2 4 4 2 4 0]]


In [ ]:
# Data Model
# input to the OR-Tools routing solver
def create_data_model():
    """Prepare the data dictionary required by OR-Tools."""
    data = {
        'distance_matrix': duration_matrix_minutes.tolist(),
        'num_vehicles': num_buses,
        'vehicle_capacities': vehicle_capacities,
        'demands': [0] + [1] * num_students,  # 0 demand for school, 1 per student
        'depot': 0  # school is both start & end point
    }
    return data

In [ ]:
'''It uses Google OR-Tools to:

Load your distance/time matrix and vehicle info

Tell OR-Tools how to calculate travel costs and student loads

Solve the best way to assign students to buses and in which order

Return the optimized routes'''
def optimize_routes():#Calls your previously defined create_data_model()
    data = create_data_model()
    #translator between your own location indices and internal indices
    manager = pywrapcp.RoutingIndexManager(len(data['distance_matrix']),
                                           data['num_vehicles'],
                                           data['depot'])
    #engine that builds and solves your route optimization problem.
    routing = pywrapcp.RoutingModel(manager)

    #how to calculate the travel cost (in your case, duration in minutes) between any two locations.
    def distance_callback(from_idx, to_idx):
        from_node = manager.IndexToNode(from_idx)
        to_node = manager.IndexToNode(to_idx)
        return data['distance_matrix'][from_node][to_node]

    #cost function
    #Registers the distance_callback with the routing model.
    transit_cb_idx = routing.RegisterTransitCallback(distance_callback)
    routing.SetArcCostEvaluatorOfAllVehicles(transit_cb_idx)
    #"For every route segment (arc), calculate its cost using this function."

    #tells OR-Tools how much load or demand is picked up at each location.
    def demand_callback(from_idx):
        from_node = manager.IndexToNode(from_idx)
        return data['demands'][from_node]

    # how to handle vehicle capacity constraints
    demand_cb_idx = routing.RegisterUnaryTransitCallback(demand_callback)
    routing.AddDimensionWithVehicleCapacity(demand_cb_idx, 0, data['vehicle_capacities'], True, 'Capacity')

    search_params = pywrapcp.DefaultRoutingSearchParameters()
    #PATH_CHEAPEST_ARC – Greedy nearest-neighbor
    search_params.first_solution_strategy = routing_enums_pb2.FirstSolutionStrategy.PATH_CHEAPEST_ARC
    solution = routing.SolveWithParameters(search_params)
    if solution:
        routes = print_solution(data, manager, routing, solution)
        return routes
    else:
        print("No solution found!")
        return []

In [ ]:
# Print Solution
def print_solution(data, manager, routing, solution):
    all_routes = []  # collect ordered routes for later mapping

    for vehicle_id in range(data['num_vehicles']):
        index = routing.Start(vehicle_id)
        route = []  # ordered list of stops for this bus
        plan_output = []  # string output for this bus's route
        route_distance = 0
        route_load = 0

        plan_output.append(f"🚍 Route for bus {vehicle_id}:")

        # Traverse each stop assigned to this bus
        while not routing.IsEnd(index):
            node_index = manager.IndexToNode(index)  # map routing index to node index
            route_load += data['demands'][node_index]
            plan_output.append(f"  - Stop {node_index}: cumulative load {route_load}")
            route.append(node_index)  # store the ordered stop index
            prev_index = index
            index = solution.Value(routing.NextVar(index))  # move to next stop
            route_distance += routing.GetArcCostForVehicle(prev_index, index, vehicle_id)

        # Add school at the end of the route
        plan_output.append("  - School (end of route)")
        plan_output.append(f"Route distance: {route_distance} min")
        plan_output.append(f"Total load on route: {route_load}")

        print("\n".join(plan_output))  # print the route details
        all_routes.append(route)  # save the route sequence

    return all_routes

In [ ]:
# Run Optimizer
routes = optimize_routes()

🚍 Route for bus 0:
  - Stop 0: cumulative load 0
  - Stop 2: cumulative load 1
  - Stop 5: cumulative load 2
  - Stop 7: cumulative load 3
  - Stop 6: cumulative load 4
  - Stop 9: cumulative load 5
  - School (end of route)
Route distance: 6 min
Total load on route: 5
🚍 Route for bus 1:
  - Stop 0: cumulative load 0
  - Stop 8: cumulative load 1
  - Stop 3: cumulative load 2
  - Stop 4: cumulative load 3
  - Stop 10: cumulative load 4
  - Stop 1: cumulative load 5
  - School (end of route)
Route distance: 6 min
Total load on route: 5


In [ ]:
# Visualize routes and stops on an interactive Folium map
map_center = [school_lat, school_lon]
bus_map = folium.Map(location=map_center, zoom_start=15)

colors = ['blue', 'green', 'purple', 'orange', 'cadetblue']

folium.Marker(
    location=[school_lat, school_lon],
    popup="School",
    icon=folium.Icon(color='red', icon='graduation-cap', prefix='fa')
).add_to(bus_map)

for i, (lat, lon) in enumerate(student_locations, 1):
    folium.Marker(
        location=[lat, lon],
        popup=f"Student {i}",
        icon=folium.Icon(color='gray', icon='user', prefix='fa')
    ).add_to(bus_map)

for bus_idx, route in enumerate(routes):
    route_coords = [locations[0]]
    route_coords += [locations[i] for i in route if i != 0]
    route_coords.append(locations[0])

    folium.PolyLine(
        locations=route_coords,
        color=colors[bus_idx % len(colors)],
        weight=4,
        opacity=0.8,
        popup=f"Bus {bus_idx} Route"
    ).add_to(bus_map)

# Display the map in the notebook
bus_map